# AI-Based Network Intrusion Detection and Threat Analysis System (AI-NIDS)
## Academic Term Paper Experiments: *AI in Cybersecurity and Defense: Risks, Benefits, and Frameworks*
---
### Authors & Project Scope
This research notebook provides an end-to-end reproducible machine learning pipeline evaluating:
- **Model A:** Logistic Regression (Linear Baseline)
- **Model B:** Random Forest Classifier (Tree Ensemble)
- **Model C:** XGBoost Classifier (Gradient-Boosted Decision Trees)

**Key Research Focus:**
1. Multi-metric evaluation highlighting **False Negative Rate (FNR)** and **F1-Score** over naive accuracy.
2. Investigation of class imbalance mitigation techniques.
3. Explainable AI (XAI) feature attribution distinguishing correlation from causality.
4. Computational latency vs detection capability trade-off analysis on macOS / Apple Silicon.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.settings import DEFAULT_RANDOM_STATE, DEFAULT_TEST_SIZE, SAMPLE_DATA_DIR, RAW_DATA_DIR
from src.data_loader import DataLoader
from src.preprocessing import PreprocessingPipeline
from src.model_trainer import ModelTrainer, XGBOOST_AVAILABLE
from src.evaluation import calculate_comprehensive_metrics, compile_comparison_table, explain_metric_significance
from src.explainability import ModelExplainer
from src.sample_data import generate_cicids_benchmark_sample
print("Environment initialized successfully. XGBoost available:", XGBOOST_AVAILABLE)

## Section 1: Dataset Ingestion & Binarization
We ingest the CICIDS2017 flow dataset, detect the  column, and map multi-class attack categories to binary (, ).

In [ ]:
sample_csv = SAMPLE_DATA_DIR / "CICIDS2017_Benchmark_Sample.csv"
if not sample_csv.exists():
    generate_cicids_benchmark_sample(n_samples=5000, output_filepath=sample_csv)

loader = DataLoader()
raw_df = loader.load_csv(sample_csv)
df, orig_labels = loader.binarize_labels(raw_df)
print(f"Loaded {len(df):,} records with {len(df.columns)} columns.")
print("Class distribution:", df["Label"].value_counts().to_dict())

## Section 2: Data Preprocessing & Leakage Prevention
To prevent data leakage in security research, imputation and scaling transformations are fitted **exclusively on the training partition** after stratified splitting.

In [ ]:
pipeline = PreprocessingPipeline(test_size=DEFAULT_TEST_SIZE, random_state=DEFAULT_RANDOM_STATE)
X_train, X_test, y_train, y_test, feature_names = pipeline.fit_transform(df)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Retained features: {len(feature_names)}")
print("Audit summary:", pipeline.get_summary()["class_distribution"])

## Section 3: Model Training (Logistic Regression, Random Forest, XGBoost)
We train all models under identical train/test conditions and record wall-clock training times.

In [ ]:
trainer = ModelTrainer(random_state=DEFAULT_RANDOM_STATE)
models = ["logistic_regression", "random_forest"]
if XGBOOST_AVAILABLE:
    models.append("xgboost")

metrics_results = []
for model_key in models:
    model, t_train = trainer.train_model(model_key, X_train, y_train, handle_imbalance=True)
    y_pred, y_proba, t_pred = trainer.predict(model_key, X_test)
    m = calculate_comprehensive_metrics(y_test, y_pred, y_proba, t_train, t_pred, model_name=model_key)
    metrics_results.append(m)

comp_df = compile_comparison_table(metrics_results)
comp_df

## Section 4: Experiment 2 — Empirical Impact of Class Imbalance
We compare an unweighted Random Forest against a cost-sensitive balanced Random Forest.

In [ ]:
_, t_u = trainer.train_model("random_forest", X_train, y_train, handle_imbalance=False)
y_pred_u, y_prob_u, t_p_u = trainer.predict("random_forest", X_test)
m_unweighted = calculate_comprehensive_metrics(y_test, y_pred_u, y_prob_u, t_u, t_p_u, model_name="RF (Unweighted)")

_, t_b = trainer.train_model("random_forest", X_train, y_train, handle_imbalance=True)
y_pred_b, y_prob_b, t_p_b = trainer.predict("random_forest", X_test)
m_balanced = calculate_comprehensive_metrics(y_test, y_pred_b, y_prob_b, t_b, t_p_b, model_name="RF (Balanced Weights)")

compile_comparison_table([m_unweighted, m_balanced])

## Section 5: Explainable AI & Feature Attribution
We inspect the top 15 features contributing to tree splits.

In [ ]:
rf_model = trainer.trained_models["random_forest"]
explainer = ModelExplainer(model=rf_model, feature_names=feature_names)
top_features = explainer.get_feature_importance(top_n=15)
top_features[["Rank", "Feature", "Relative_Percentage", "Cybersecurity_Context"]]

## Section 6: Discussion & Academic Term Paper Conclusions
1. **Accuracy is Insufficient:** Relying solely on accuracy fails under class imbalance; False Negative Rate (FNR) is the most critical metric in cyber defense.
2. **Ensemble Utility:** Random Forest and XGBoost excel at capturing nonlinear traffic threshold signatures (packet size ratios, inter-arrival time bursts).
3. **Correlation vs Causality:** Features with high split importance reflect statistical artifacts of attack tools, not physical causation.